##Comment Toxicity Detection using NLP, Machine Learning, and Transformers

##Headings

1. Import Libraries
2. Load Dataset
3. Exploratory Data Analysis
4. Text Preprocessing
5. Feature Engineering
6. Model Building
7. Model Evaluation
8. Model Comparison
9. Save Best Model

##1. Import Libraries

In [ ]:
#Imports

# Data Manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Text PreProcessing
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

#Custom Transformer

from sklearn.base import BaseEstimator, TransformerMixin

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

# Evaluation Metrics
from sklearn.metrics import classification_report,confusion_matrix

# Machine Learning Models
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

#Transformer Imports
import torch

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)

from datasets import Dataset


In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

##2. Load Dataset

In [ ]:
df_train = pd.read_csv(
    '/content/train (5).csv',
    engine='python',
    on_bad_lines='skip'
)

print(df_train.shape)

In [ ]:
df_train

In [ ]:
df_train.head()

In [ ]:
df_train.info()

In [ ]:
df_train.columns

#Checking for Null Values

In [ ]:
df_train.isnull().sum()

##Checking for Duplicates

In [ ]:
df_train.duplicated().sum()

In [ ]:
df_train = df_train.drop_duplicates()

In [ ]:
df_train.duplicated().sum()

##2.Exploratory Data Analysis

In [ ]:
#Creating a Binary column 'Label' for EDA Purpose:

df_train['label'] = df_train[
    ['toxic', 'severe_toxic', 'obscene',
     'threat', 'insult', 'identity_hate']
].max(axis=1)

df_train['label_name'] = df_train['label'].map({
    0: 'Non-toxic',
    1: 'Toxic'
})


In [ ]:
df_train[['label','label_name']]

In [ ]:
#Class Distribution

df_train['label_name'].value_counts()

In [ ]:
#Visualization Of Class Distribution

plt.figure(figsize=(6,4))

df_train['label_name'].value_counts().plot(
    kind='bar',
    color=['green', 'red']
)

plt.title("Class Imbalance")
plt.xlabel("Class")
plt.ylabel("Count")

plt.savefig("class_imbalance.png")
plt.show()

Observation from class distributions:

*   The dataset is highly imbalanced, with non-toxic comments
significantly outnumbering toxic comments.
*   Approximately 89.8% of comments are non-toxic, while only 10.2% are toxic.

In [ ]:
#Class Imbalance Percentage

df_train['label_name'].value_counts(normalize=True) * 100

Observation from class imbalance percentage:

*   The dataset contains a severe class imbalance. The majority class is non-toxic comments, indicating that special attention should be given to evaluation metrics such as F1-score and recall during model evaluation.



In [ ]:
#Comment Length

df_train['comment_length'] = df_train['comment_text'].apply(len)

In [ ]:
#Comment Lenghth Distribution

df_train['comment_length']

In [ ]:
#Comment Length Distribution Plot

plt.figure(figsize=(8,5))

sns.histplot(df_train['comment_length'], bins=50)

plt.title("Distribution of Comment Length")

plt.savefig("comment_length_distribution.png")
plt.show()

Observation from Comment Lenghth Distribution:



---



*   Most comments in the dataset are relatively short in length, with only a small number of extremely long comments present.
*   The distribution is right-skewed, indicating the presence of outliers with very large comment lengths.

In [ ]:
#Comment Length by Toxicity

plt.figure(figsize=(8,5))

sns.boxplot(
    x='label_name',
    y='comment_length',
    data=df_train
)

plt.title("Comment Length by Toxicity")

plt.savefig("comment_length_toxicity.png")
plt.show()

Observation from Comment Lenghth by Toxicity:

---

*   Toxic comments tend to have slightly shorter median lengths compared to non-toxic comments.
*    However, both categories contain several long outlier comment

In [ ]:
#Most Common Words

from collections import Counter

all_words = " ".join(df_train['comment_text']).split()

common_words = Counter(all_words).most_common(20)

common_words

Observation from Most Common Words:

*   The most common words mainly consist of stopwords such as “the”, “to”, and “of”.
*   This indicates that text preprocessing steps like stopword removal will be important before model training.



In [ ]:
#Toxic Words

toxic_comments = df_train[df_train['label'] == 1]

toxic_words = " ".join(toxic_comments['comment_text']).split()

Counter(toxic_words).most_common(20)

Observation from Toxic Words:



*   The toxic comments still contain many common English stopwords because preprocessing has not yet been applied.
*   More meaningful toxic keywords are expected to appear after text cleaning and stopword removal.

In [ ]:
from wordcloud import WordCloud

wordcloud = WordCloud(
    width=800,
    height=400,
    background_color='black'
).generate(" ".join(df_train['comment_text']))

plt.figure(figsize=(12,6))

plt.imshow(wordcloud)

plt.axis("off")

plt.savefig("wordcloud.png")

plt.show()

Observation from Word Cloud :

*   The WordCloud visualization highlights the most frequently occurring words in the dataset.
*   Terms such as “article”, “Wikipedia”, and “edit” appear prominently because the dataset is derived from Wikipedia discussion comments.

In [ ]:
#Downloading all plots

from google.colab import files

files.download("class_imbalance.png")
files.download("comment_length_distribution.png")
files.download("comment_length_toxicity.png")
files.download("wordcloud.png")

##4. Custom Transformer for text Preprocessing

Text Preprocessing Techniques:

1. Lowercasing
2. Remove URLs
3. Remove special characters
4. Remove numbers
5. Remove extra spaces
6. Tokenization
7. Stopword removal
8. Lemmatization

In [ ]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
#Custom Transformer

class TextPreprocessor(BaseEstimator, TransformerMixin):
   def __init__(self):
     self.stop_word = set(stopwords.words('english'))
     self.lemmatizer = WordNetLemmatizer()
     self.word_tokenize = word_tokenize

   def text_cleaning(self,text):
      text=str(text).lower()
      #remove links
      text= re.sub(r'https?://\S+|www\.\S+',' ',text)
      #remove hashtags
      text=re.sub(r'#\w+',' ',text)
      #remove mentions
      text = re.sub(r'@\w+',' ',text)
      #mail-id
      text=re.sub(r'\S+@\S+',' ',text)
      text=re.sub(r'[^a-z0-9\s]',' ',text)
      #remove whitespaces
      text = re.sub(r'\s+', ' ', text).strip()
      return text
   def tokenize_lemmatize(self,text):
      tokens=self.word_tokenize(text)
      return ' '.join([self.lemmatizer.lemmatize(word) for word in tokens if word not in self.stop_word])

   def fit(self,X,Y=None):
      return self

   def transform(self, X, y=None):
      return [self.tokenize_lemmatize(self.text_cleaning(text)) for text in X]


In [ ]:
#Testing Text Preprocessor

sample_text = [
    "YOU are an IDIOT!!! Visit https://abc.com now!!!"
]

preprocessor = TextPreprocessor()

preprocessor.transform(sample_text)

##Splitting X and Y

In [ ]:
X = df_train['comment_text']
y = df_train[
    [
        'toxic',
        'severe_toxic',
        'obscene',
        'threat',
        'insult',
        'identity_hate'
    ]
]

#Splitting X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

##1. Multinomial NB Model

In [ ]:
NB_pipeline = Pipeline([

    ('PreProcess', TextPreprocessor()),

    ('TF-IDF', TfidfVectorizer()),

    ('Multinomial_Model',
     OneVsRestClassifier(MultinomialNB()))
])

In [ ]:
#Model Training

NB_pipeline.fit(X_train, y_train)

In [ ]:
#Prediction

y_pred_nb = NB_pipeline.predict(X_test)

In [ ]:
#Evaluation for MultinomialNB Classifier

target_names = [
    'toxic',
    'severe_toxic',
    'obscene',
    'threat',
    'insult',
    'identity_hate'
]

print(
    classification_report(
        y_test,
        y_pred_nb,
        target_names=target_names
    )
)

#Multinomial NB Model

Observation from MultinomialNB Model



*   The Multinomial Naive Bayes model achieved poor performance on minority toxicity categories such as severe_toxic, threat, and identity_hate.

*   Although the model showed relatively high precision for some classes, recall values were extremely low, indicating that the classifier failed to identify many toxic comments.

*   This behavior is primarily caused by the severe class imbalance present in the dataset.



##2. Logistic Regression Model

In [ ]:
LR_pipeline = Pipeline([

    ('PreProcess', TextPreprocessor()),

    ('TF-IDF', TfidfVectorizer()),

    ('LogisticRegression_Model',
     OneVsRestClassifier(LogisticRegression(max_iter=1000)))
])

In [ ]:
#Model Training

LR_pipeline.fit(X_train, y_train)

In [ ]:
#Prediction

y_pred_lr = LR_pipeline.predict(X_test)

In [ ]:
#Evaluation for Logistic Regression Model

target_names = [
    'toxic',
    'severe_toxic',
    'obscene',
    'threat',
    'insult',
    'identity_hate'
]

print(
    classification_report(
        y_test,
        y_pred_lr,
        target_names=target_names
    )
)
#Logistic Regression Model

Observation from LogisticRegression Model :



*   The Logistic Regression model significantly outperformed the Multinomial Naive Bayes classifier across most toxicity categories.
*   The model achieved strong performance on frequently occurring labels such as toxic, obscene, and insult, while performance remained relatively weak for rare labels like threat and identity_hate due to severe class imbalance.



Insights :

Linear models such as Logistic Regression often perform strongly in NLP tasks because TF-IDF generates high-dimensional sparse feature vectors that are well suited for linear classifiers.




Why Logistic Regression is best ?

Logistic Regression achieved the best balance between precision, recall, and F1-score across most toxicity categories.

##3. Linear Support Vector Classifier

In [ ]:
SVC_pipeline = Pipeline([

    ('PreProcess', TextPreprocessor()),

    ('TF-IDF', TfidfVectorizer()),

    ('LinearSVC_Model',
     OneVsRestClassifier(LinearSVC()))

])

In [ ]:
#Model Training

SVC_pipeline.fit(X_train, y_train)

In [ ]:
#Prediction

y_pred_svc = SVC_pipeline.predict(X_test)

In [ ]:
target_names = [
    'toxic',
    'severe_toxic',
    'obscene',
    'threat',
    'insult',
    'identity_hate'
]

print(
    classification_report(
        y_test,
        y_pred_svc,
        target_names=target_names
    )
)
#Linear SVC Model

Observation from Linear SVC:



*   Among the classical machine learning models evaluated, LinearSVC achieved the best overall performance for multi-label toxicity classification using TF-IDF features.

*   The LinearSVC model outperformed both Multinomial Naive Bayes and Logistic Regression across most toxicity categories.
*    The model achieved the highest micro-average and weighted-average F1-scores, demonstrating strong capability in handling high-dimensional sparse textual data generated through TF-IDF vectorization.
*   Although the model performed well on frequently occurring labels such as toxic and obscene, performance remained comparatively lower for rare categories like threat and identity_hate due to severe class imbalance in the dataset.



##4. Experimenting BERT Transformer Model :


In [ ]:
!pip install transformers datasets accelerate -q

In [ ]:
#Creating New DataFrames

train_df = pd.DataFrame({
    'text': X_train,
})

train_df = pd.concat(
    [train_df, y_train.reset_index(drop=True)],
    axis=1
)

test_df = pd.DataFrame({
    'text': X_test,
})

test_df = pd.concat(
    [test_df, y_test.reset_index(drop=True)],
    axis=1
)

In [ ]:
#Convert to HuggingFace Dataset

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [ ]:
#Loading Tokenizer

tokenizer = BertTokenizer.from_pretrained(
    'bert-base-uncased'
)

In [ ]:
#Tokenization Function

def tokenize_function(example):

  text = str(example["text"])

  return tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=128
    )

In [ ]:
train_dataset = train_dataset.map(tokenize_function)
test_dataset = test_dataset.map(tokenize_function)

In [ ]:
label_columns = [
    'toxic',
    'severe_toxic',
    'obscene',
    'threat',
    'insult',
    'identity_hate'
]

def format_labels(example):

    example['labels'] = [
        float(example[label]) if example[label] is not None else 0.0
        for label in label_columns
    ]

    return example

In [ ]:
train_dataset = train_dataset.map(format_labels)
test_dataset = test_dataset.map(format_labels)

In [ ]:
#Removing Unnecessary COlumns
train_dataset = train_dataset.remove_columns(
    ['text'] + label_columns
)

test_dataset = test_dataset.remove_columns(
    ['text'] + label_columns
)

In [ ]:
train_dataset.set_format('torch')
test_dataset.set_format('torch')

In [ ]:
#Loading BERT Model

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=6,
    problem_type='multi_label_classification'
)

In [ ]:
training_args = TrainingArguments(

    output_dir='./results',

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=1,

    eval_strategy='epoch',

    save_strategy='epoch',

    logging_steps=100
)

In [ ]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset
)

In [ ]:
trainer.train()

In [ ]:
#Saving Logistic Regression Model

import pickle

with open('logistic_regression_pipeline.pkl', 'wb') as file:
    pickle.dump(LR_pipeline, file)

print("Logistic Regression model saved successfully!")

In [ ]:
#Saving Linear SVC Model

import pickle

# Save Linear SVC Pipeline
with open('linear_svc_pipeline.pkl', 'wb') as file:
    pickle.dump(SVC_pipeline, file)

print("Linear SVC model saved successfully!")

In [ ]:
#Saving BERT Transformer Model

trainer.save_model("bert_toxicity_model")

tokenizer.save_pretrained("bert_toxicity_model")

print("BERT model saved successfully!")

In [ ]:
#Downloading Pickle FIles

from google.colab import files

files.download('logistic_regression_pipeline.pkl')
files.download('linear_svc_pipeline.pkl')

In [ ]:
#Downloading BERT Model

!zip -r bert_toxicity_model.zip bert_toxicity_model

files.download('bert_toxicity_model.zip')

In [ ]:
##########################################  DONE  ##############################################